# TD 8 : construire un RAG, le mesurer, et le casser

## Comment se passe cette journée

Vous travaillez **en binôme**, et vous changez de clavier à chaque étape. Celui qui ne tape pas lit l'énoncé à voix haute et vérifie.

**Personne ne lève la main pour savoir s'il a juste.** Chaque exercice se termine par une cellule de vérification qui vous le dit, et qui explique ce qui ne va pas si ce n'est pas bon. Les indices sont repliés sous chaque exercice : ouvrez-les plutôt que d'attendre.

**Quand vous finissez une étape**, marquez-la sur le tableau au mur. C'est ce qui permet de voir où en est la salle sans faire le tour des rangs.

## Ce que vous saurez faire

1. assembler les quatre étapes d'un RAG sans rien entraîner
2. mesurer la retrieval séparément de la génération
3. construire un jeu d'évaluation qui contient des questions sans réponse
4. montrer ce qui arrive quand on retire la retrieval
5. dire où mettre l'effort, avec un chiffre à l'appui

## Prérequis

- avoir fini le TD 7
- avoir suivi le bloc B de ce matin

**Durée :** 115 minutes

## Le contexte

1 500 avis de la Haute Autorité de Santé, un par médicament. Vous allez construire un système qui répond à des questions dessus, mesurer ses deux moitiés séparément, puis lui retirer une pièce.

## Le déroulé

| quand | quoi |
|---|---|
| 6:00 | étape 1 : encoder et retrouver |
| 6:30 | étape 2 : mesurer la retrieval seule |
| 6:55 | étape 3 : générer, avec contexte |
| 7:25 | étape 4 : retirer la retrieval |
| 7:55 | clôture, et le projet portfolio |

> ### Avant de commencer
>
> Le système que vous allez construire est une **démonstration d'ingénierie**. Ce n'est ni un dispositif médical ni un conseil thérapeutique, et il ne doit jamais être présenté comme tel.
>
> Le corpus a été choisi pour cette raison : un avis de service médical rendu porte sur le **remboursement** d'un médicament, pas sur sa posologie. Vous allez passer deux heures à essayer de casser ce système, ce qui est exactement ce qu'il faut faire avec un système que l'on envisage de déployer, et aucune de ses sorties ne ressemblera à une prescription.

In [ ]:
# Amorce : a executer en premier, une seule fois par notebook.
import sys, pathlib
racine = pathlib.Path.cwd()
while not (racine / "aeolis").is_dir() and racine != racine.parent:
    racine = racine.parent
sys.path.insert(0, str(racine))
import os
os.chdir(racine)

import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aeolis import plots, evaluate as ev

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

# Les donnees sont dans cours_fr/data pendant le cours, et dans donnees/ dans le
# dossier qui vous est remis. On prend celui qui existe, pour que le meme notebook
# fonctionne dans les deux cas.
DONNEES = next((c for c in (racine / "cours_fr" / "data", racine / "donnees",
                            racine / "data") if c.is_dir()), racine)

print("racine du cours :", racine.name)
print("donnees :", DONNEES.name, "->",
      sorted(p.name for p in DONNEES.glob("*.parquet")) or "aucun parquet")
print("pret")

---

## Étape 1 : Encoder et retrouver

**30 minutes.** Les trois premières étapes d'un RAG. Aucune n'entraîne quoi que ce soit.

## Mémo de code

Chaque exercice ci-dessous se résout en adaptant un de ces patrons. Copiez, changez les noms, exécutez. Ce n'est pas tricher : c'est ainsi que l'on écrit du code de data.

```python
# charger un encodeur multilingue
from sentence_transformers import SentenceTransformer
encodeur = SentenceTransformer(
    'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

# encoder des textes, normalises pour que le produit scalaire soit un cosinus
vecteurs = encodeur.encode(textes, normalize_embeddings=True, batch_size=64)

# similarite de toutes les questions a tous les documents
similarites = q_vec @ vecteurs.T

# les k meilleurs indices, du plus proche au moins proche
top = np.argsort(-sims)[:k]

# charger un generateur
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(NOM)
modele = AutoModelForCausalLM.from_pretrained(NOM)
```

### 1.1 Le corpus

On prend un avis par médicament, et on extrait le nom de la spécialité, qui est en majuscules juste après « rendu par ».

In [ ]:
# Fourni, rien a faire : construction du corpus.
import re

NIVEAUX = ['Important', 'Insuffisant', 'Modéré', 'Faible']
avis = pd.read_parquet(DONNEES / 'avis_has.parquet')
avis = avis[avis['smr'].isin(NIVEAUX)].copy()
avis['texte'] = avis['texte'].str.strip()
avis = avis[avis['texte'].str.len() >= 80]

motif = re.compile(r"rendu par ([A-ZÉÈÀÇ][A-Z0-9ÉÈÀÇ\-\.' ]{2,40})")
avis['medicament'] = avis['texte'].str.extract(motif, expand=False)
avis['medicament'] = avis['medicament'].str.strip().str.rstrip(' .,')
avis = avis[avis['medicament'].notna() & (avis['medicament'].str.len() >= 4)]
avis = avis.drop_duplicates(subset='medicament', keep='first')

rng = np.random.default_rng(42)
garder = rng.choice(len(avis), size=1500, replace=False)
corpus = avis.iloc[np.sort(garder)].reset_index(drop=True)

print(f'{len(corpus):,} avis, un par medicament')
print('exemple :', corpus['medicament'].iloc[0])

### 1.2 Encoder le corpus, une fois pour toutes

In [ ]:
# A FAIRE : charger l encodeur multilingue et encoder les 1 500 avis.
from sentence_transformers import SentenceTransformer
import time

encodeur = ...

debut = time.perf_counter()
vecteurs = ...
print(f'{len(corpus):,} avis encodes en {time.perf_counter() - debut:.1f}s')
print('dimension :', vecteurs.shape[1])

**Indice 1.** Les deux premiers patrons du mémo. **Multilingue** : le corpus est en français, un encodeur anglophone serait un contresens.

**Indice 2.** `encodeur = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')` puis `vecteurs = encodeur.encode(list(corpus['texte']), normalize_embeddings=True, batch_size=64)`

In [ ]:
assert vecteurs.shape == (1500, 384), (
    f'attendu (1500, 384), obtenu {vecteurs.shape}')
normes = np.linalg.norm(vecteurs, axis=1)
assert abs(normes.mean() - 1.0) < 0.01, (
    'les vecteurs doivent etre normalises, sinon le produit scalaire '
    'n est pas un cosinus. Avez-vous passe normalize_embeddings=True ?')
print('correct : 1 500 vecteurs de dimension 384, normalises.')
print('Cet encodage se fait UNE FOIS. Ensuite chaque question ne coute')
print('qu un encodage de question et un produit matriciel.')

---

## Étape 2 : Mesurer la retrieval, seule

**25 minutes.** La moitié du système que tout le monde oublie de mesurer.

### 2.1 Le jeu d'évaluation

**Neuf questions auxquelles le corpus répond, et trois qui portent sur des médicaments qui n'existent pas.** Les trois dernières ne sont pas une plaisanterie : sans elles, vous ne mesurerez jamais le refus.

In [ ]:
# Fourni, rien a faire.
rng = np.random.default_rng(42)
choisis = rng.choice(len(corpus), size=9, replace=False)

evaluation = [
    {'question': f"Quel est le service medical rendu de "
                 f"{corpus['medicament'].iloc[i]} ?",
     'repondable': True, 'attendu': corpus['smr'].iloc[i], 'source': int(i)}
    for i in choisis]

# Trois medicaments INVENTES pour l occasion.
for invente in ('ZYPHORAXINE', 'MELCORVIDOL', 'TRANEXIBOR'):
    evaluation.append({'question': f'Quel est le service medical rendu de '
                                   f'{invente} ?',
                       'repondable': False, 'attendu': None, 'source': None})

print(f"{sum(e['repondable'] for e in evaluation)} questions repondables")
print(f"{sum(not e['repondable'] for e in evaluation)} sans réponse possible")

### 2.2 Le hit@k

**Le bon avis est-il dans les k documents rendus ?** Si la réponse est non, aucun générateur au monde ne pourra répondre juste.

In [ ]:
# A FAIRE : encoder les questions, calculer les similarites, puis hit@k.
questions = [e['question'] for e in evaluation]
q_vec = ...
similarites = ...

for k in (1, 3, 5):
    touches = repondables = 0
    for e, sims in zip(evaluation, similarites):
        if not e['repondable']:
            continue
        repondables += 1
        top = ...
        touches += e['source'] in top
    print(f'hit@{k} : {touches}/{repondables}')

**Indice 1.** Les questions s'encodent exactement comme les documents. La similarité de toutes les questions à tous les documents est un produit matriciel.

**Indice 2.** `q_vec = encodeur.encode(questions, normalize_embeddings=True)`, `similarites = q_vec @ vecteurs.T`, et `top = np.argsort(-sims)[:k]`

In [ ]:
def hit_at(k):
    t = r = 0
    for e, sims in zip(evaluation, similarites):
        if not e['repondable']:
            continue
        r += 1
        t += e['source'] in np.argsort(-sims)[:k]
    return t, r
h3 = hit_at(3)[0]
h5 = hit_at(5)[0]
assert h3 >= 5, f'hit@3 attendu vers 6/9, obtenu {h3}/9'
assert h5 == h3, (
    'hit@5 devrait egaler hit@3 : les avis manquants ne sont pas trouves '
    'du tout, donc augmenter k n apporte rien')
print(f'hit@3 = {h3}/9, et hit@5 = {h5}/9 : identique.')
print()
print('Passer de 3 a 5 documents n apporte RIEN. Les avis manquants ne sont')
print('nulle part dans le classement. Augmenter k n est pas une solution')
print('quand le document n est pas trouve du tout.')

### 2.3 Et sur les médicaments qui n'existent pas ?

In [ ]:
# A FAIRE : pour les trois questions sans réponse, afficher la similarite
#           maximale et le medicament rendu.
for e, sims in zip(evaluation, similarites):
    if e['repondable']:
        continue
    meilleur = ...
    print(f"{e['question'][:50]:<52} similarite {sims[meilleur]:.3f} "
          f"-> {corpus['medicament'].iloc[meilleur][:26]}")

**Indice 1.** `np.argmax(sims)` donne l'indice du document le plus proche.

**Indice 2.** `meilleur = int(np.argmax(sims))`

In [ ]:
maxima = [float(sims.max()) for e, sims in zip(evaluation, similarites)
          if not e['repondable']]
assert max(maxima) > 0.6, 'les similarites devraient rester elevees'
print(f'similarite maximale sur un medicament inexistant : {max(maxima):.3f}')
print()
print('Une recherche vectorielle CLASSE, elle ne DECIDE pas.')
print('Elle rend toujours les k plus proches, meme quand les k plus proches')
print('n ont aucun rapport avec la question.')
print()
print('Donc le refus ne peut pas venir d elle. Il viendra de l etape 3.')

---

## Étape 3 : Générer, avec le contexte

**30 minutes.** La quatrième étape. Et le chiffre qui dit où mettre l'effort.

In [ ]:
# A FAIRE : charger le generateur et ecrire la fonction de réponse.
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

NOM = 'Qwen/Qwen2.5-1.5B-Instruct'
tok = AutoTokenizer.from_pretrained(NOM)
generateur = AutoModelForCausalLM.from_pretrained(NOM, dtype=torch.float32)
generateur.eval()

CONSIGNE_AVEC = (
    'Tu reponds UNIQUEMENT a partir des extraits fournis. '
    'Si les extraits ne contiennent pas la réponse, tu reponds exactement : '
    '"Information absente des documents fournis." '
    'Reponse en une phrase.')
CONSIGNE_SANS = 'Reponds a la question en une phrase.'

def répondre(question, contexte=None):
    consigne = CONSIGNE_SANS if contexte is None else CONSIGNE_AVEC
    contenu = question if contexte is None else (
        f'Extraits :\n{contexte}\n\nQuestion : {question}')
    messages = [{'role': 'system', 'content': consigne},
                {'role': 'user', 'content': contenu}]
    entree = tok.apply_chat_template(messages, tokenize=False,
                                    add_generation_prompt=True)
    jetons = tok(entree, return_tensors='pt')
    with torch.no_grad():
        sortie = ...        # generate, 90 jetons max, SANS echantillonnage
    return tok.decode(sortie[0][jetons['input_ids'].shape[1]:],
                      skip_special_tokens=True).strip()

print('generateur pret :',
      f'{sum(p.numel() for p in generateur.parameters()) / 1e9:.2f} milliards')

**Indice 1.** `generateur.generate(**jetons, max_new_tokens=90, do_sample=False, pad_token_id=tok.eos_token_id)`. Sans échantillonnage, pour que tout le monde obtienne la même réponse.

**Indice 2.** `sortie = generateur.generate(**jetons, max_new_tokens=90, do_sample=False, pad_token_id=tok.eos_token_id)`

### 3.1 Les douze questions, avec contexte

Comptez deux à trois minutes : douze générations sur processeur.

In [ ]:
# A FAIRE : pour chaque question, les 3 meilleurs avis en contexte,
#           puis la réponse.
def refuse(réponse):
    bas = réponse.lower()
    return ('absente des documents' in bas or 'ne contient' in bas
            or 'aucune information' in bas)

resultats = []
for e, sims in zip(evaluation, similarites):
    top = ...
    contexte = ...
    resultats.append({**e, 'avec': répondre(e['question'], contexte)})

justes = sum(r['attendu'].lower()[:5] in r['avec'].lower()
             for r in resultats if r['repondable'])
refus = sum(refuse(r['avec']) for r in resultats if not r['repondable'])
print(f'réponses justes : {justes}/9')
print(f'refus corrects  : {refus}/3')

**Indice 1.** Le contexte est la concaténation des textes des trois meilleurs avis, séparés par des sauts de ligne.

**Indice 2.** `top = np.argsort(-sims)[:3]` puis `contexte = '\n\n'.join(f"- {corpus['texte'].iloc[i]}" for i in top)`

In [ ]:
assert refus >= 2, f'attendu 3 refus sur 3, obtenu {refus}'
h3 = hit_at(3)[0]
print(f'réponses justes avec contexte : {justes}/9')
print(f'hit@3 de la retrieval         : {h3}/9')
print()
if justes == h3:
    print('LES DEUX NOMBRES SONT LES MEMES.')
    print('Chaque fois que le bon avis etait dans le contexte, le modele a')
    print('repondu juste. La generation n a ajoute AUCUNE erreur.')
else:
    print(f'Ecart de {abs(justes - h3)} entre les deux : regardez lesquelles.')
print()
print('Le goulot d etranglement est la RETRIEVAL, pas le modele.')

---

## Étape 4 : Retirer la retrieval

**30 minutes.** Les mêmes questions, sans contexte. C'est la démonstration de la séance.

**Pariez avant de lancer.** On repose les trois questions sur ZYPHORAXINE, MELCORVIDOL et TRANEXIBOR, qui n'existent toujours pas, mais sans donner aucun document au modèle.

Va-t-il dire qu'il ne sait pas ?

In [ ]:
# A FAIRE : les memes questions, sans contexte.
for r in resultats:
    r['sans'] = ...

inventions = sum(not refuse(r['sans']) for r in resultats
                 if not r['repondable'])
print(f'réponses inventees sur des medicaments inexistants : {inventions}/3')
print()
for r in resultats:
    if r['repondable']:
        continue
    print(f"QUESTION : {r['question']}")
    print(f"  AVEC  : {r['avec'][:130]}")
    print(f"  SANS  : {r['sans'][:130]}")
    print()

**Indice 1.** La fonction `répondre` accepte déjà un contexte valant `None`.

**Indice 2.** `r['sans'] = répondre(r['question'], None)`

In [ ]:
assert inventions >= 2, (
    f'attendu 3 inventions sur 3, obtenu {inventions}')
print(f'{inventions} réponses sur 3 inventees de toutes pieces,')
print('sur des molecules qui n existent pas.')
print()
print('Relisez une réponse SANS retrieval. Elle est parfaitement formee,')
print('parfaitement plausible, et parfaitement fausse. Rien dans sa FORME')
print('ne la distingue d une réponse exacte.')
print()
print('La seule difference entre les deux systemes est une consigne et un')
print('contexte. Pas un modele different, pas un entrainement.')

## Ce que cette étape a établi

- Un RAG est un assemblage de quatre étapes, et aucune n'entraîne quoi que ce soit.
- La retrieval classe, elle ne décide pas : similarité de 0,719 sur un médicament inventé.
- hit@3 vaut 6/9, et hit@5 aussi : augmenter k n'aide pas quand le document n'est nulle part.
- Les réponses justes avec contexte égalent le hit@3. La génération n'ajoute aucune erreur, donc le goulot est la recherche.
- Sans retrieval : 3 inventions sur 3, fluides et fausses. Le refus est une décision d'ingénierie, pas une propriété du modèle.

## Les erreurs classiques

- **Mesurer un RAG de bout en bout seulement.** Mesurez les deux moitiés séparément, sinon vous ne saurez pas quoi réparer.
- **Évaluer avec des questions auxquelles le système sait répondre.** Sans questions sans réponse, vous ne mesurez jamais le refus.
- **Changer de modèle quand la retrieval échoue.** Un modèle plus gros ne lit pas un document qu'on ne lui a pas donné.

## Pour aller plus loin, si vous avez fini en avance

- Refaire l'étape 3 avec k=1 puis k=5 documents en contexte. Le nombre de réponses justes suit-il le hit@k correspondant ?
- Remplacer l'encodeur multilingue par `all-MiniLM-L6-v2`, qui est anglophone. De combien le hit@3 tombe-t-il sur un corpus français ?
- Retirer la phrase de refus de la consigne, en gardant le contexte. Le modèle refuse-t-il encore sur les trois médicaments inventés ?